# Reproduce the E5 complete two-stage comparison

Open this notebook in the extracted package directory. Read `E5_Report.md`
and `source/PROTOCOL.md` first. All data are reused; the study is a post-inspection
extension, not independent confirmation. Training is optional; replay is the
default and does not change any weights.


In [ ]:
from pathlib import Path
import json, subprocess, sys
ROOT = Path.cwd()
assert (ROOT / "source/two_stage.py").exists(), "Open from the package root"
print(sys.version)
print((ROOT / "E5_Report.md").read_text())


Install the requirements into your Python environment if needed: `python -m pip install -r requirements.txt`. The experiment used Python 3.12.14 and CPU PyTorch 2.10.0. Saved observations are supplied and are not regenerated.

In [ ]:
subprocess.run([sys.executable, "analyze.py"], cwd=ROOT, check=True)
audit = json.loads((ROOT / "results/audit.json").read_text())
assert audit["status"] == "passed" and audit["checkpoint_replays"] == 210
print(audit["status"], audit["checkpoint_replays"])


In [ ]:
import pandas as pd
s = pd.read_csv(ROOT / "results/paired_summary.csv")
s[(s.numerator == "P3") & (s.metric == "forecast_mse")]


## Optional fresh training

Set `RETRAIN=True` to run all 210 field fits in a separate output directory. Each method reuses its fixed A1/A2 delay and the original paired field initialization. No test score selects settings. The retained results are not overwritten.

In [ ]:
RETRAIN = False
if RETRAIN:
    subprocess.run([sys.executable, "source/two_stage.py", "--mode", "validate", "--output", "rerun_results"], cwd=ROOT, check=True)
    subprocess.run([sys.executable, "source/two_stage.py", "--mode", "full", "--workers", "8", "--output", "rerun_results"], cwd=ROOT, check=True)
    subprocess.run([sys.executable, "analyze.py", "--results", "rerun_results"], cwd=ROOT, check=True)


Checkpoint replay and source hashes check consistency. They do not constitute independent scientific replication, validate the mathematical proofs, or establish journal readiness.